# psygnal

## Example 1: a Signal on a class

In [ ]:
# Example 1: a Signal declared on a class, connected callbacks receive the emitted arguments
from psygnal import Signal

class Thermometer:
    changed = Signal(float, str)

received = []
thermo = Thermometer()
thermo.changed.connect(lambda value, unit: received.append((value, unit)))
thermo.changed.emit(21.5, "C")
thermo.changed.emit(70.7, "F")
assert received == [(21.5, "C"), (70.7, "F")]
assert len(thermo.changed) == 1  # one connected callback
print("Example 1:", received)

## Example 2: disconnect, blocking and pausing

In [ ]:
# Example 2: blocked() drops emissions, paused() queues and replays them, disconnect removes a callback
from psygnal import Signal

class Counter:
    ticked = Signal(int)

seen = []
counter = Counter()
counter.ticked.connect(seen.append)
with counter.ticked.blocked():
    counter.ticked.emit(1)
with counter.ticked.paused():
    counter.ticked.emit(2)
    counter.ticked.emit(3)
    assert seen == []  # held until the pause ends
counter.ticked.disconnect(seen.append)
counter.ticked.emit(4)
assert seen == [2, 3], seen
print("Example 2:", seen)

## Example 3: an evented dataclass

In [ ]:
# Example 3: psygnal.evented gives each dataclass field a signal under obj.events
import dataclasses
import psygnal

@psygnal.evented
@dataclasses.dataclass
class Settings:
    name: str = "run"
    epochs: int = 1

changes = []
settings = Settings()
settings.events.epochs.connect(lambda new, old: changes.append(("epochs", old, new)))
settings.events.connect(lambda info: changes.append(("any", info.signal.name)))
settings.epochs = 5
settings.name = "long-run"
assert ("epochs", 1, 5) in changes and ("any", "name") in changes, changes
print("Example 3:", changes)

## Example 4: evented containers

In [ ]:
# Example 4: EventedList and EventedDict emit inserted / removed / changed signals
from psygnal.containers import EventedDict, EventedList

log = []
items = EventedList([1, 2])
items.events.inserted.connect(lambda index, value: log.append(("inserted", index, value)))
items.events.removed.connect(lambda index, value: log.append(("removed", index, value)))
items.append(3)
items.pop(0)
config = EventedDict({"lr": 0.1})
config.events.changed.connect(lambda key, old, new: log.append(("changed", key, old, new)))
config["lr"] = 0.01
assert log == [("inserted", 2, 3), ("removed", 0, 1), ("changed", "lr", 0.1, 0.01)], log
print("Example 4:", list(items), dict(config), log)

## Example 5: debounced and throttled callbacks

In [ ]:
# Example 5: debounced() calls once after a burst settles; throttled() calls at most once per interval
import time
from psygnal import debounced, throttled

calls = {"debounced": [], "throttled": []}
on_settled = debounced(lambda x: calls["debounced"].append(x), timeout=50)
on_rate = throttled(lambda x: calls["throttled"].append(x), timeout=200, leading=True)
for i in range(10):
    on_settled(i)
    on_rate(i)
time.sleep(0.5)
assert calls["debounced"] == [9], calls  # only the last call of the burst
assert calls["throttled"][0] == 0 and len(calls["throttled"]) <= 2, calls
print("Example 5:", calls)